# Pokemon Pipeline — Exploración de Datos Raw
### Python para Data Engineers · DataHackers Academy
---
**Objetivo:** Ver los datos *exactamente como vienen* de las fuentes, antes de cualquier transformación.

El trabajo del Data Engineer empieza aquí: **entender el problema antes de codear la solución.**

## 0. Setup

In [1]:
import sys
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')
sys.path.append('..')   # para importar src/

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Librerias cargadas')
print(f'pandas version: {pd.__version__}')

Librerias cargadas
pandas version: 3.0.2


## 1. comparacion.csv — datos sintéticos con suciedad intencional
Leemos con `dtype=str` para ver los datos **exactamente como vienen**.  
Sin que pandas intente inferir tipos — eso enmascararía los problemas.

In [2]:
from src.extract import extract_comparaciones

df_raw = extract_comparaciones()

print(f'Shape: {df_raw.shape}')
print(f'Dtypes: {df_raw.dtypes.to_dict()}')

[extract] comparacion.csv -> 510 filas, 4 columnas
Shape: (510, 4)
Dtypes: {'pokemon_a': <StringDtype(na_value=nan)>, 'nivel_a': <StringDtype(na_value=nan)>, 'pokemon_b': <StringDtype(na_value=nan)>, 'nivel_b': <StringDtype(na_value=nan)>}


In [3]:
# Primeras filas — todo es string, nulos incluidos
df_raw.head(10)

,pokemon_a,nivel_a,pokemon_b,nivel_b
0,7.0,95.0,63.0,-1.0
1,7.0,58.0,140.0,1.0
2,87.0,45.0,98.0,6.0
3,93.0,30.0,18.0,11.0
4,42.0,90.0,54.0,83.0
5,70.0,42.0,143.0,100.0
6,146.0,64.0,81.0,83.0
7,150.0,29.0,150.0,18.0
8,109.0,60.0,0.0,71.0
9,88.0,1.0,112.0,93.0


In [4]:
# Nulos por columna
nulos     = df_raw.isna().sum()
nulos_pct = (df_raw.isna().mean() * 100).round(2)

pd.DataFrame({'nulos': nulos, 'porcentaje': nulos_pct}).sort_values('porcentaje', ascending=False)

,nulos,porcentaje
pokemon_a,18,3.53
pokemon_b,13,2.55
nivel_a,9,1.76
nivel_b,7,1.37


In [5]:
# Duplicados exactos
dupes = df_raw.duplicated().sum()
print(f'Duplicados exactos: {dupes}')
print()
df_raw[df_raw.duplicated(keep=False)].head(6)

Duplicados exactos: 10



,pokemon_a,nivel_a,pokemon_b,nivel_b
28,92.0,46.0,64.0,53.0
46,55.0,0.0,87.0,54.0
155,77.0,69.0,130.0,73.0
187,26.0,41.0,-1.0,150.0
205,85.0,84.0,111.0,18.0
273,8.0,56.0,43.0,13.0


In [6]:
# Convertir a numerico para detectar valores fuera de rango
df_num = df_raw.apply(pd.to_numeric, errors='coerce')

mask_pokemon_invalido = (
    (df_num['pokemon_a'] < 1) | (df_num['pokemon_a'] > 150) |
    (df_num['pokemon_b'] < 1) | (df_num['pokemon_b'] > 150)
)
mask_nivel_invalido = (
    (df_num['nivel_a'] < 1) | (df_num['nivel_a'] > 100) |
    (df_num['nivel_b'] < 1) | (df_num['nivel_b'] > 100)
)

print(f'Filas con pokemon fuera de rango (< 1 o > 150): {mask_pokemon_invalido.sum()}')
print(f'Filas con nivel fuera de rango  (< 1 o > 100): {mask_nivel_invalido.sum()}')
print()
print('Ejemplos:')
df_raw[mask_pokemon_invalido | mask_nivel_invalido].head(8)

Filas con pokemon fuera de rango (< 1 o > 150): 49
Filas con nivel fuera de rango  (< 1 o > 100): 41

Ejemplos:


,pokemon_a,nivel_a,pokemon_b,nivel_b
0,7.0,95.0,63.0,-1.0
8,109.0,60.0,0.0,71.0
10,77.0,20.0,130.0,101.0
27,29.0,48.0,151.0,19.0
35,142.0,49.0,108.0,101.0
37,132.0,-1.0,86.0,40.0
38,19.0,0.0,148.0,52.0
45,200.0,51.0,137.0,85.0


In [7]:
# Resumen de calidad del CSV
total       = len(df_raw)
solo_nulos  = df_raw.isna().any(axis=1).sum()
solo_fuera  = (mask_pokemon_invalido | mask_nivel_invalido).sum()
solo_dupes  = df_raw.duplicated().sum()
mask_sucios = df_raw.isna().any(axis=1) | mask_pokemon_invalido | mask_nivel_invalido | df_raw.duplicated()

print('=== Resumen de calidad ===')
print(f'Total filas:              {total}')
print(f'Filas con nulos:          {solo_nulos}  ({solo_nulos/total*100:.1f}%)')
print(f'Filas fuera de rango:     {solo_fuera}  ({solo_fuera/total*100:.1f}%)')
print(f'Duplicados exactos:       {solo_dupes}  ({solo_dupes/total*100:.1f}%)')
print(f'Total filas con suciedad: {mask_sucios.sum()}  ({mask_sucios.sum()/total*100:.1f}%)')

=== Resumen de calidad ===
Total filas:              510
Filas con nulos:          47  (9.2%)
Filas fuera de rango:     87  (17.1%)
Duplicados exactos:       10  (2.0%)
Total filas con suciedad: 133  (26.1%)


In [8]:
# Distribucion de niveles en filas validas
niveles_validos = df_num[['nivel_a', 'nivel_b']].stack().dropna()
niveles_validos = niveles_validos[(niveles_validos >= 1) & (niveles_validos <= 100)]

print('Distribucion de niveles validos:')
print(niveles_validos.describe())

Distribucion de niveles validos:
count   962.00
mean     52.48
std      28.45
min       1.00
25%      27.25
50%      54.00
75%      76.75
max     100.00
dtype: float64


## 2. pokemon.json — datos de la PokeAPI
Generados por `generate_data.py`. Vienen limpios de la API en **formato JSON** — no hay suciedad que corregir.
A diferencia de `comparacion.csv`, los tipos ya son nativos (int, None) directamente desde la API.

In [9]:
import json

# Estructura raw del JSON antes de cargarlo al DataFrame
with open('../data/raw/pokemon.json', 'r', encoding='utf-8') as f:
    pokemon_raw = json.load(f)

print(f'Tipo de dato: {type(pokemon_raw)}')
print(f'Total registros: {len(pokemon_raw)}')
print()
print('Primer registro (estructura completa):')
print(json.dumps(pokemon_raw[0], indent=2))

Tipo de dato: <class 'list'>
Total registros: 150

Primer registro (estructura completa):
{
  "id": 1,
  "name": "bulbasaur",
  "type1": "grass",
  "type2": "poison",
  "hp": 45,
  "attack": 49,
  "defense": 49,
  "special_attack": 65,
  "special_defense": 65,
  "speed": 45,
  "effort_hp": 0,
  "effort_attack": 0,
  "effort_defense": 0,
  "effort_special_attack": 1,
  "effort_special_defense": 0,
  "effort_speed": 0,
  "height": 7,
  "weight": 69,
  "base_experience": 64
}


In [10]:
# En JSON, type2 ausente se representa como null (None en Python)
sin_type2 = [p for p in pokemon_raw if p['type2'] is None]
print(f'Pokemon sin tipo secundario (type2: null en JSON): {len(sin_type2)}')
print()
print('Ejemplo de registro con type2 nulo:')
print(json.dumps(sin_type2[0], indent=2))

Pokemon sin tipo secundario (type2: null en JSON): 83

Ejemplo de registro con type2 nulo:
{
  "id": 4,
  "name": "charmander",
  "type1": "fire",
  "type2": null,
  "hp": 39,
  "attack": 52,
  "defense": 43,
  "special_attack": 60,
  "special_defense": 50,
  "speed": 65,
  "effort_hp": 0,
  "effort_attack": 0,
  "effort_defense": 0,
  "effort_special_attack": 0,
  "effort_special_defense": 0,
  "effort_speed": 1,
  "height": 6,
  "weight": 85,
  "base_experience": 62
}


In [11]:
from src.extract import extract_pokemon

# extract_pokemon() lee pokemon.json y lo convierte a DataFrame
df_pokemon = extract_pokemon()

print(f'Shape: {df_pokemon.shape}')
print(f'Columnas: {list(df_pokemon.columns)}')
print()
# A diferencia del CSV (todo object/str), el JSON preserva tipos nativos
print('Dtypes (tipos nativos del JSON):')
print(df_pokemon.dtypes)

[extract] pokemon.json -> 150 filas, 19 columnas
Shape: (150, 19)
Columnas: ['id', 'name', 'type1', 'type2', 'hp', 'attack', 'defense', 'special_attack', 'special_defense', 'speed', 'effort_hp', 'effort_attack', 'effort_defense', 'effort_special_attack', 'effort_special_defense', 'effort_speed', 'height', 'weight', 'base_experience']

Dtypes (tipos nativos del JSON):
id                        int64
name                        str
type1                       str
type2                       str
hp                        int64
attack                    int64
defense                   int64
special_attack            int64
special_defense           int64
speed                     int64
effort_hp                 int64
effort_attack             int64
effort_defense            int64
effort_special_attack     int64
effort_special_defense    int64
effort_speed              int64
height                    int64
weight                    int64
base_experience           int64
dtype: object


In [12]:
# Primeras filas — todo string, tal como sale del CSV
df_pokemon.head(10)

,id,name,type1,type2,hp,attack,defense,special_attack,special_defense,speed,effort_hp,effort_attack,effort_defense,effort_special_attack,effort_special_defense,effort_speed,height,weight,base_experience
0,1,bulbasaur,grass,poison,45,49,49,65,65,45,0,0,0,1,0,0,7,69,64
1,2,ivysaur,grass,poison,60,62,63,80,80,60,0,0,0,1,1,0,10,130,142
2,3,venusaur,grass,poison,80,82,83,100,100,80,0,0,0,2,1,0,20,1000,236
3,4,charmander,fire,NaN,39,52,43,60,50,65,0,0,0,0,0,1,6,85,62
4,5,charmeleon,fire,NaN,58,64,58,80,65,80,0,0,0,1,0,1,11,190,142
5,6,charizard,fire,flying,78,84,78,109,85,100,0,0,0,3,0,0,17,905,240
6,7,squirtle,water,NaN,44,48,65,50,64,43,0,0,1,0,0,0,5,90,63
7,8,wartortle,water,NaN,59,63,80,65,80,58,0,0,1,0,1,0,10,225,142
8,9,blastoise,water,NaN,79,83,100,85,105,78,0,0,0,0,3,0,16,855,239
9,10,caterpie,bug,NaN,45,30,35,20,20,45,1,0,0,0,0,0,3,29,39


In [13]:
# Tipos de datos y nulos
print('Dtypes:')
print(df_pokemon.dtypes)
print()
print('Nulos por columna:')
print(df_pokemon.isna().sum())

Dtypes:
id                        int64
name                        str
type1                       str
type2                       str
hp                        int64
attack                    int64
defense                   int64
special_attack            int64
special_defense           int64
speed                     int64
effort_hp                 int64
effort_attack             int64
effort_defense            int64
effort_special_attack     int64
effort_special_defense    int64
effort_speed              int64
height                    int64
weight                    int64
base_experience           int64
dtype: object

Nulos por columna:
id                         0
name                       0
type1                      0
type2                     83
hp                         0
attack                     0
defense                    0
special_attack             0
special_defense            0
speed                      0
effort_hp                  0
effort_attack              0
ef

In [14]:
# Distribucion de tipos (type1)
print('Pokemon por tipo primario:')
print(df_pokemon['type1'].value_counts())

Pokemon por tipo primario:
type1
water       28
normal      22
poison      14
grass       12
fire        12
bug         12
electric     9
rock         9
ground       8
fighting     7
psychic      7
ghost        3
dragon       3
fairy        2
ice          2
Name: count, dtype: int64


In [15]:
# Cuantos tienen tipo secundario (type2)?
tiene_type2 = df_pokemon['type2'].notna().sum()
print(f'Pokemon con tipo secundario: {tiene_type2} ({tiene_type2/len(df_pokemon)*100:.1f}%)')
print(f'Pokemon sin tipo secundario: {df_pokemon["type2"].isna().sum()}')

Pokemon con tipo secundario: 67 (44.7%)
Pokemon sin tipo secundario: 83


In [16]:
# Stats base en formato raw (string) — convertir para ver distribucion
STAT_COLS = ['hp', 'attack', 'defense', 'special_attack', 'special_defense', 'speed']

df_stats = df_pokemon[STAT_COLS].apply(pd.to_numeric, errors='coerce')
print('Estadisticas base (raw -> numeric):')
df_stats.describe().round(1)

Estadisticas base (raw -> numeric):


,hp,attack,defense,special_attack,special_defense,speed
count,150.00,150.00,150.00,150.00,150.00,150.00
mean,64.00,72.70,68.00,66.90,65.90,68.90
std,28.50,26.80,26.90,28.50,24.10,27.00
min,10.00,5.00,5.00,15.00,20.00,15.00
25%,45.00,50.50,50.00,45.00,48.50,45.80
50%,60.00,70.00,65.00,63.00,65.00,69.00
75%,79.80,91.50,82.20,85.00,80.00,90.00
max,250.00,134.00,180.00,154.00,125.00,150.00


In [17]:
# Los 5 mas pesados y mas altos (valores raw de la API)
print('Top 5 mas pesados (weight en hectogramos — raw):')
print(df_pokemon[['name', 'weight']].assign(weight=df_pokemon['weight'].astype(float))
      .sort_values('weight', ascending=False).head(5).to_string(index=False))
print()
print('Top 5 mas altos (height en decimetros — raw):')
print(df_pokemon[['name', 'height']].assign(height=df_pokemon['height'].astype(float))
      .sort_values('height', ascending=False).head(5).to_string(index=False))

Top 5 mas pesados (weight en hectogramos — raw):
     name  weight
  snorlax 4600.00
    golem 3000.00
 gyarados 2350.00
   lapras 2200.00
dragonite 2100.00

Top 5 mas altos (height en decimetros — raw):
     name  height
     onix   88.00
 gyarados   65.00
dragonair   40.00
    arbok   35.00
   lapras   25.00
